# Street Network Validation

Visual validation of the OSM projection-based street graph built from Cyclomedia recording metadata.

**Method:** Recordings snapped to OSM drive network edges, projected onto edge geometry, subsampled at ~25m intervals along each edge. Intersections connected via shared OSM nodes.

**What this notebook checks:**
1. Degree distribution — mostly degree 2 (mid-block), 3-4 at intersections
2. Edge distance distribution — should cluster around ~25m
3. Connected components — should be mostly one component
4. Close-up map — edges follow road geometry, no cross-block connections
5. Yaw correctness — all values in [0, 360)
6. Spatial coverage — uniform sampling vs raw recordings
7. Face resolution — F/R/B/L map to sensible neighbors
8. Random walk navigability

In [ ]:
import sys
from pathlib import Path

# Add project root to path so pickle can resolve dagspaces classes
PROJECT_ROOT = str(Path("../..").resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pickle
import math
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

try:
    import folium
    from folium.plugins import MarkerCluster
    FOLIUM_OK = True
except ImportError:
    FOLIUM_OK = False
    print("folium not installed — map cells will use matplotlib fallback")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120

# ---------- Load graph ----------
GRAPH_PATH = "../../data/cyclomedia/street_graph_osm_25m.pkl"

with open(GRAPH_PATH, "rb") as f:
    graph = pickle.load(f)

n_nodes = len(graph.adjacency)
n_edges = sum(len(v) for v in graph.adjacency.values())
print(f"Nodes: {n_nodes:,}")
print(f"Directed edges: {n_edges:,}  (undirected: {n_edges // 2:,})")
print(f"Avg degree: {n_edges / n_nodes:.2f}")

In [ ]:
# Pre-compute arrays used by every section
rec_ids = np.array(list(graph.adjacency.keys()))
lats = np.array([graph.coords[r][0] for r in rec_ids])
lons = np.array([graph.coords[r][1] for r in rec_ids])
yaws = np.array([graph.yaw_degrees.get(r, 0.0) for r in rec_ids])
degrees = np.array([len(graph.adjacency[r]) for r in rec_ids])

# Classify nodes
is_isolated = degrees == 0
is_endpoint = degrees == 1
is_midblock = degrees == 2
is_intersection = degrees > 2

print(f"Isolated (deg 0):    {is_isolated.sum():>6,}")
print(f"Endpoints (deg 1):   {is_endpoint.sum():>6,}")
print(f"Mid-block (deg 2):   {is_midblock.sum():>6,}")
print(f"Intersection (deg>2):{is_intersection.sum():>6,}")

## 1. Degree Distribution

A well-formed street network should be mostly degree 2 (mid-block), with degree 3-4 at intersections and degree 1 at dead ends / road termini.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

deg_counts = Counter(degrees)
max_deg = max(deg_counts.keys())
x_vals = list(range(max_deg + 1))
y_vals = [deg_counts.get(d, 0) for d in x_vals]

# Bar chart
ax = axes[0]
colors = ["#d62728" if d == 0 else "#ff7f0e" if d == 1 else "#2ca02c" if d == 2 else "#1f77b4" for d in x_vals]
ax.bar(x_vals, y_vals, color=colors, edgecolor="white", linewidth=0.5)
ax.set_xlabel("Node degree")
ax.set_ylabel("Count")
ax.set_title("Degree distribution")
ax.set_yscale("log")
ax.set_xlim(-0.5, min(max_deg, 10) + 0.5)

# Cumulative
ax = axes[1]
sorted_deg = np.sort(degrees)
cdf = np.arange(1, len(sorted_deg) + 1) / len(sorted_deg)
ax.plot(sorted_deg, cdf, linewidth=2)
ax.set_xlabel("Degree")
ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF of node degree")
ax.axhline(0.95, color="gray", linestyle="--", alpha=0.5, label="95%")
ax.legend()

plt.tight_layout()
plt.show()

for d in sorted(deg_counts.keys()):
    print(f"  degree {d}: {deg_counts[d]:>6,} nodes  ({100 * deg_counts[d] / n_nodes:5.1f}%)")

## 2. Edge Distance Distribution

Mid-block edges should be ~25m (the subsampling target). Continuation edges bridge segment boundaries (~50-120m). Intersection edges connect across roads (~20-60m). Anything over ~200m is suspicious.

In [ ]:
all_dists = np.array([nb.distance_m for nbs in graph.adjacency.values() for nb in nbs])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(all_dists, bins=100, range=(0, 300), color="#1f77b4", edgecolor="white", linewidth=0.3)
ax.axvline(25, color="red", linestyle="--", alpha=0.7, label="25m target")
ax.axvline(np.median(all_dists), color="orange", linestyle="--", alpha=0.7, label=f"median {np.median(all_dists):.0f}m")
ax.set_xlabel("Edge distance (m)")
ax.set_ylabel("Count")
ax.set_title("Edge distance distribution")
ax.legend()

ax = axes[1]
ax.hist(all_dists, bins=100, range=(0, 300), cumulative=True, density=True,
        histtype="step", linewidth=2, color="#1f77b4")
ax.set_xlabel("Edge distance (m)")
ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF of edge distances")
for thresh in [50, 100, 200]:
    frac = np.mean(all_dists <= thresh)
    ax.axvline(thresh, color="gray", linestyle=":", alpha=0.5)
    ax.text(thresh + 2, frac - 0.05, f"{frac:.0%} ≤ {thresh}m", fontsize=8)

plt.tight_layout()
plt.show()

print(f"Edge distance stats:")
print(f"  Mean:   {np.mean(all_dists):>7.1f} m")
print(f"  Median: {np.median(all_dists):>7.1f} m")
print(f"  P5:     {np.percentile(all_dists, 5):>7.1f} m")
print(f"  P95:    {np.percentile(all_dists, 95):>7.1f} m")
print(f"  Max:    {np.max(all_dists):>7.1f} m")
print(f"  > 200m: {np.sum(all_dists > 200):,} edges ({100 * np.mean(all_dists > 200):.1f}%)")

## 3. Connected Components

The network should be mostly one large component. Small isolated components may represent short dead-end streets or data gaps.

In [ ]:
# Compute connected components
rid_to_idx = {r: i for i, r in enumerate(rec_ids)}
comp_labels = np.full(n_nodes, -1, dtype=int)
comp_id = 0
for start_idx in range(n_nodes):
    if comp_labels[start_idx] >= 0:
        continue
    queue = [start_idx]
    while queue:
        idx = queue.pop()
        if comp_labels[idx] >= 0:
            continue
        comp_labels[idx] = comp_id
        for nb in graph.adjacency.get(rec_ids[idx], []):
            nb_idx = rid_to_idx.get(nb.recording_id)
            if nb_idx is not None and comp_labels[nb_idx] < 0:
                queue.append(nb_idx)
    comp_id += 1

comp_sizes = Counter(comp_labels)
sorted_comps = comp_sizes.most_common()
largest_id, largest_size = sorted_comps[0]

print(f"Total components: {comp_id}")
print(f"Largest component: {largest_size:,} nodes ({100 * largest_size / n_nodes:.1f}%)")
print(f"\nTop 10 components:")
for cid, sz in sorted_comps[:10]:
    print(f"  Component {cid}: {sz:,} nodes")

fig, ax = plt.subplots(figsize=(8, 3))
sizes_arr = np.array([sz for _, sz in sorted_comps])
ax.bar(range(min(30, len(sizes_arr))), sizes_arr[:30], color="#1f77b4")
ax.set_xlabel("Component rank")
ax.set_ylabel("Size (nodes)")
ax.set_title("Connected component sizes (top 30)")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## 5. Intersection Close-Up

Zoom into a specific area (default: Times Square) to verify that intersection nodes appear where roads cross, edges follow road geometry, and yaw arrows point along the road.

In [ ]:
# --- Configuration: pick an area to inspect ---
FOCUS_LAT, FOCUS_LON = 40.758, -73.9855   # Times Square
FOCUS_RADIUS_DEG = 0.004                   # ~400m

# Filter to local nodes
local_mask = (np.abs(lats - FOCUS_LAT) < FOCUS_RADIUS_DEG) & (np.abs(lons - FOCUS_LON) < FOCUS_RADIUS_DEG)
local_idx = np.where(local_mask)[0]
local_rids = set(rec_ids[local_idx])
print(f"Local nodes: {len(local_idx)} (within {FOCUS_RADIUS_DEG}° of ({FOCUS_LAT}, {FOCUS_LON}))")
print(f"  Mid-block: {is_midblock[local_idx].sum()}, Intersection: {is_intersection[local_idx].sum()}, "
      f"Endpoint: {is_endpoint[local_idx].sum()}")

In [ ]:
if FOLIUM_OK:
    m2 = folium.Map(location=[FOCUS_LAT, FOCUS_LON], zoom_start=17, tiles="CartoDB positron")

    # Draw only edges where BOTH endpoints are local (no ghost lines flying off-screen)
    for i in local_idx:
        rid = rec_ids[i]
        lat_i, lon_i = graph.coords[rid]
        for nb in graph.adjacency[rid]:
            if nb.recording_id not in local_rids:
                continue  # skip edges to nodes outside the viewport
            lat_j, lon_j = graph.coords[nb.recording_id]
            # Color by type: red if cross-heading (intersection edge), blue if same-heading
            yaw_diff = abs(graph.yaw_degrees.get(rid, 0) - graph.yaw_degrees.get(nb.recording_id, 0)) % 360
            is_same_heading = yaw_diff < 40 or yaw_diff > 320
            color = "#1f77b4" if is_same_heading else "#d62728"
            weight = 2 if is_same_heading else 3
            folium.PolyLine([(lat_i, lon_i), (lat_j, lon_j)],
                            weight=weight, color=color, opacity=0.7).add_to(m2)

    # Draw nodes with yaw arrows
    arrow_len = 0.00015  # ~15m in degrees
    for i in local_idx:
        rid = rec_ids[i]
        lat_i, lon_i = lats[i], lons[i]
        yaw_i = yaws[i]
        deg = degrees[i]

        # Node circle
        if is_intersection[i]:
            color, radius = "#d62728", 6
        elif is_endpoint[i]:
            color, radius = "#ff7f0e", 5
        elif is_isolated[i]:
            color, radius = "#999999", 5
        else:
            color, radius = "#1f77b4", 4

        folium.CircleMarker(
            [lat_i, lon_i], radius=radius, color=color, fill=True, fill_opacity=0.8, weight=1,
            popup=f"<b>{rid}</b><br>deg={deg}<br>yaw={yaw_i:.1f}°",
        ).add_to(m2)

        # Yaw arrow (short line in the heading direction)
        yaw_rad = math.radians(yaw_i)
        dlat = arrow_len * math.cos(yaw_rad)
        dlon = arrow_len * math.sin(yaw_rad) / math.cos(math.radians(lat_i))
        folium.PolyLine(
            [(lat_i, lon_i), (lat_i + dlat, lon_i + dlon)],
            weight=2, color=color, opacity=0.9,
        ).add_to(m2)

    m2.save("network_closeup_map.html")
    print("Saved: network_closeup_map.html")
    print("Blue edges = same-heading (along-road), Red edges = cross-heading (intersection)")
    display(m2)
else:
    fig, ax = plt.subplots(figsize=(10, 10))
    for i in local_idx:
        rid = rec_ids[i]
        for nb in graph.adjacency[rid]:
            if nb.recording_id not in local_rids:
                continue
            nb_coord = graph.coords.get(nb.recording_id)
            if nb_coord:
                ax.plot([lons[i], nb_coord[1]], [lats[i], nb_coord[0]],
                        "k-", alpha=0.3, linewidth=0.5)
    ax.scatter(lons[local_idx][is_midblock[local_idx]], lats[local_idx][is_midblock[local_idx]],
               s=10, c="steelblue", label="mid-block", zorder=3)
    ax.scatter(lons[local_idx][is_intersection[local_idx]], lats[local_idx][is_intersection[local_idx]],
               s=40, c="red", marker="^", label="intersection", zorder=4)
    ax.scatter(lons[local_idx][is_endpoint[local_idx]], lats[local_idx][is_endpoint[local_idx]],
               s=20, c="orange", label="endpoint", zorder=3)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"Close-up: ({FOCUS_LAT}, {FOCUS_LON})")
    ax.legend()
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

## 6. Yaw Distribution

All yaw values should be in [0, 360). The distribution should reflect Manhattan's grid (~29°/209° for avenues, ~119°/299° for cross-streets).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
ax = axes[0]
ax.hist(yaws, bins=72, range=(0, 360), color="#1f77b4", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Yaw (degrees)")
ax.set_ylabel("Count")
ax.set_title("Yaw distribution (all nodes)")
# Mark Manhattan grid directions
for angle, label in [(29, "Ave NE"), (209, "Ave SW"), (119, "XSt SE"), (299, "XSt NW")]:
    ax.axvline(angle, color="red", linestyle="--", alpha=0.5)
    ax.text(angle + 2, ax.get_ylim()[1] * 0.9, label, fontsize=7, color="red", rotation=90, va="top")

# Polar / compass rose
ax = axes[1]
ax_polar = fig.add_subplot(122, projection="polar")
bins = np.linspace(0, 360, 17)  # 16 bins of 22.5°
counts, _ = np.histogram(yaws, bins=bins)
theta = np.radians((bins[:-1] + bins[1:]) / 2)
width = np.radians(360 / 16)
bars = ax_polar.bar(theta, counts, width=width, bottom=0, alpha=0.7, color="#1f77b4", edgecolor="white")
ax_polar.set_theta_zero_location("N")
ax_polar.set_theta_direction(-1)
ax_polar.set_title("Yaw compass rose", pad=20)
axes[1].set_visible(False)  # hide the cartesian axes behind polar

plt.tight_layout()
plt.show()

print(f"Yaw range: [{yaws.min():.1f}, {yaws.max():.1f}]")
print(f"In [0, 360): {np.sum((yaws >= 0) & (yaws < 360)):,} / {len(yaws):,}")

## 7. Spatial Coverage vs Raw Recordings

Compare the subsampled graph nodes against the full set of raw recordings to verify coverage is uniform and no large areas are missing.

In [ ]:
import pandas as pd

raw_df = pd.read_parquet("../../data/cyclomedia/manhattan_2025_1_1k_scratch.parquet",
                         columns=["recording_id", "lat", "lon"])
raw_recs = raw_df.drop_duplicates(subset=["recording_id"])
raw_lats = pd.to_numeric(raw_recs["lat"], errors="coerce").dropna().values
raw_lons = pd.to_numeric(raw_recs["lon"], errors="coerce").dropna().values

fig, axes = plt.subplots(1, 2, figsize=(14, 18), sharex=True, sharey=True)

ax = axes[0]
ax.scatter(raw_lons, raw_lats, s=0.05, alpha=0.15, c="gray")
ax.set_title(f"Raw recordings ({len(raw_lats):,})")
ax.set_ylabel("Latitude")
ax.set_xlabel("Longitude")
ax.set_aspect("equal")

ax = axes[1]
ax.scatter(raw_lons, raw_lats, s=0.02, alpha=0.05, c="gray", label="raw")
ax.scatter(lons, lats, s=1.5, alpha=0.5, c="steelblue", label=f"graph ({n_nodes:,})")
ax.scatter(lons[is_intersection], lats[is_intersection], s=6, c="red",
           label=f"intersections ({is_intersection.sum():,})", zorder=4)
ax.set_title(f"Graph nodes overlaid on raw")
ax.set_xlabel("Longitude")
ax.legend(markerscale=4, loc="upper right")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

print(f"Raw recordings: {len(raw_lats):,}")
print(f"Graph nodes:    {n_nodes:,} ({100 * n_nodes / len(raw_lats):.1f}% of raw)")
print(f"Subsampling ratio: 1:{len(raw_lats) // n_nodes}")

## 8. Face Resolution Spot-Check

Pick a few intersection nodes and verify that `resolve_face_to_neighbor` maps F/R/B/L to sensible neighbors (neighbors in the expected compass direction given the node's yaw).

In [ ]:
# Pick 5 intersection nodes and test face resolution
intersection_rids = rec_ids[is_intersection]
rng = np.random.default_rng(42)
sample_rids = rng.choice(intersection_rids, min(5, len(intersection_rids)), replace=False)

BEARING_TOL = 45.0

def compass(bearing):
    dirs = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    return dirs[int(round(bearing / 45.0)) % 8]

for rid in sample_rids:
    lat_i, lon_i = graph.coords[rid]
    yaw_i = graph.yaw_degrees[rid]
    nbs = graph.adjacency[rid]

    print(f"\n{'=' * 60}")
    print(f"Node: {rid}  lat={lat_i:.6f} lon={lon_i:.6f}  yaw={yaw_i:.1f}°  deg={len(nbs)}")
    print(f"  Neighbors:")
    for nb in nbs:
        print(f"    {nb.recording_id}: {nb.distance_m:.1f}m  bearing={nb.bearing_deg:.1f}° ({compass(nb.bearing_deg)})")

    print(f"  Face resolution (tolerance={BEARING_TOL}°):")
    for face in ["F", "R", "B", "L"]:
        result = graph.resolve_face_to_neighbor(rid, face, BEARING_TOL)
        if result:
            print(f"    {face} -> {result.recording_id}  ({result.distance_m:.0f}m, {compass(result.bearing_deg)})")
        else:
            print(f"    {face} -> None (no neighbor within tolerance)")

    # Verify: F should roughly match yaw direction
    fwd = graph.resolve_face_to_neighbor(rid, "F", BEARING_TOL)
    if fwd:
        expected_bearing = yaw_i % 360
        actual_bearing = fwd.bearing_deg
        diff = abs(expected_bearing - actual_bearing) % 360
        diff = min(diff, 360 - diff)
        ok = "OK" if diff <= BEARING_TOL else "MISMATCH"
        print(f"  Forward check: expected ~{expected_bearing:.0f}°, got {actual_bearing:.0f}° (diff={diff:.0f}°) [{ok}]")

## 9. Simulated Random Walk

Run a short random walk on the graph (no VLM, just random face selection) to verify the network is navigable and walks don't immediately die.

In [ ]:
from dagspaces.urbanroamvqa.graph.street_graph import _face_for_bearing, FACE_BEARING_DEG, _normalize_bearing, _bearing_diff

def random_walk(graph, start_rid, n_steps=30, seed=0, bearing_tol=45.0):
    """Simulate a random walk on the graph, returning the trajectory."""
    rng_walk = np.random.default_rng(seed)
    trajectory = []
    rid = start_rid
    arrival_face = "F"

    for step in range(n_steps):
        lat, lon = graph.coords.get(rid, (0, 0))
        yaw = graph.yaw_degrees.get(rid, 0)
        available = [f for f in ["F", "R", "B", "L"] if f != arrival_face]
        trajectory.append({"step": step, "rid": rid, "lat": lat, "lon": lon,
                           "yaw": yaw, "arrival_face": arrival_face})

        rng_walk.shuffle(available)
        moved = False
        for face in available:
            nb = graph.resolve_face_to_neighbor(rid, face, bearing_tol)
            if nb is not None:
                trajectory[-1]["face_chosen"] = face
                trajectory[-1]["next_rid"] = nb.recording_id
                trajectory[-1]["dist_m"] = nb.distance_m
                arrival_face = graph.arrival_face(rid, nb.recording_id)
                rid = nb.recording_id
                moved = True
                break

        if not moved:
            trajectory[-1]["face_chosen"] = "STUCK"
            break

    return trajectory

# Run 5 random walks from the largest component
largest_mask = comp_labels == largest_id
largest_rids = rec_ids[largest_mask]
walk_rng = np.random.default_rng(42)
start_rids = walk_rng.choice(largest_rids, 5, replace=False)

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
all_trajs = []
walk_stats = []

for w_idx, start in enumerate(start_rids):
    traj = random_walk(graph, start, n_steps=50, seed=w_idx)
    all_trajs.append(traj)
    total_dist = sum(t.get("dist_m", 0) for t in traj)
    stuck = traj[-1].get("face_chosen") == "STUCK"
    walk_stats.append({"walk": w_idx, "steps": len(traj), "total_m": total_dist,
                        "stuck": stuck, "start": start})

# Plot each walk on its own zoomed subplot
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for w_idx, (traj, ax) in enumerate(zip(all_trajs, axes)):
    walk_lats = np.array([t["lat"] for t in traj])
    walk_lons = np.array([t["lon"] for t in traj])

    # Background: nearby graph nodes
    pad = 0.002
    mask = ((lats > walk_lats.min() - pad) & (lats < walk_lats.max() + pad) &
            (lons > walk_lons.min() - pad) & (lons < walk_lons.max() + pad))
    ax.scatter(lons[mask], lats[mask], s=1, alpha=0.2, c="gray")

    ax.plot(walk_lons, walk_lats, "-o", color=colors[w_idx], markersize=4,
            linewidth=2, alpha=0.9)
    ax.plot(walk_lons[0], walk_lats[0], "*", color=colors[w_idx], markersize=15, zorder=5)
    ax.set_title(f"Walk {w_idx}: {len(traj)} steps\n{walk_stats[w_idx]['total_m']:.0f}m", fontsize=9)
    ax.set_aspect("equal")
    ax.tick_params(labelsize=6)

plt.suptitle("Simulated random walks (50 steps max)", fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n{'Walk':<6} {'Steps':<7} {'Distance':<10} {'Stuck?':<8} {'Start'}")
for ws in walk_stats:
    print(f"{ws['walk']:<6} {ws['steps']:<7} {ws['total_m']:<10.0f} {str(ws['stuck']):<8} {ws['start']}")

## 10. Summary Statistics

In [ ]:
print("=" * 50)
print("NETWORK VALIDATION SUMMARY")
print("=" * 50)
print(f"Nodes:                {n_nodes:>10,}")
print(f"Undirected edges:     {n_edges // 2:>10,}")
print(f"Avg degree:           {n_edges / n_nodes:>10.2f}")
print(f"Max degree:           {max(degrees):>10}")
print()
print(f"Mid-block (deg 2):    {is_midblock.sum():>10,}  ({100 * is_midblock.mean():.1f}%)")
print(f"Intersection (deg>2): {is_intersection.sum():>10,}  ({100 * is_intersection.mean():.1f}%)")
print(f"Endpoint (deg 1):     {is_endpoint.sum():>10,}  ({100 * is_endpoint.mean():.1f}%)")
print(f"Isolated (deg 0):     {is_isolated.sum():>10,}  ({100 * is_isolated.mean():.1f}%)")
print()
print(f"Connected components: {comp_id:>10}")
print(f"Largest component:    {largest_size:>10,}  ({100 * largest_size / n_nodes:.1f}%)")
print()
print(f"Edge distance mean:   {np.mean(all_dists):>10.1f} m")
print(f"Edge distance median: {np.median(all_dists):>10.1f} m")
print(f"Edge distance max:    {np.max(all_dists):>10.1f} m")
print()
print(f"Yaw range:            [{yaws.min():.1f}°, {yaws.max():.1f}°]")
print(f"Yaw in [0, 360):      {np.sum((yaws >= 0) & (yaws < 360)):,} / {len(yaws):,}")
print()

# Verdict
issues = []
if is_isolated.sum() > n_nodes * 0.01:
    issues.append(f"High isolated node rate: {is_isolated.sum()} ({100*is_isolated.mean():.1f}%)")
if largest_size < n_nodes * 0.8:
    issues.append(f"Largest component only {100*largest_size/n_nodes:.0f}% — network is fragmented")
if np.max(all_dists) > 1000:
    issues.append(f"Max edge distance {np.max(all_dists):.0f}m — some edges may be non-physical")
if np.sum((yaws < 0) | (yaws >= 360)) > 0:
    issues.append(f"Yaw values outside [0,360): {np.sum((yaws < 0) | (yaws >= 360))}")

if issues:
    print("ISSUES FOUND:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("ALL CHECKS PASSED")